# Improved pairs-model analysis

Compare the naive z-score baseline with each improved model:

1. the regression model under each regime mode (`backtest_results/regression/`),
2. the Sharpe-optimized dynamic-threshold model (`backtest_results/dynamic_threshold/`),
3. the regime regression with an EWMA z-score and VIX-extreme input (`backtest_results/regime_regression/`),
4. the vol-scaled naive model (EWMA z-score, VIX size overlay, VIX no-trade band) on the original screened pairs (`backtest_results/vol_scaled_naive/original_pairs/`) and on point-in-time pairs (`backtest_results/vol_scaled_naive/`),
5. the entry-state model (trade size set at entry from spread-vol spike / high VIX) on point-in-time pairs (`backtest_results/entry_state/`).

Returns are decimals (0.05 = 5%); fold averages are across yearly test folds. Sortino is annualized against a zero minimum acceptable daily return and is calculated from daily returns.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SCENARIOS = {"0 bps": "0bps", "5 bps": "5bps"}
FOLD_KEYS = ["model_name", "pair_id", "fold_id", "test_year"]
FOLD_METRICS = [
    "total_return",
    "annualized_return",
    "sharpe_ratio",
    "sortino_ratio",
    "annualized_volatility",
    "max_drawdown",
    "average_gross_exposure",
    "average_daily_turnover",
    "total_transaction_cost",
]


def annualized_sortino(returns):
    downside_deviation = returns.clip(upper=0).pow(2).mean() ** 0.5
    if downside_deviation == 0:
        return np.nan
    return returns.mean() / downside_deviation * np.sqrt(252)


def annualized_sharpe(returns):
    volatility = returns.std()
    return returns.mean() / volatility * np.sqrt(252) if volatility > 0 else np.nan


def portfolio_row(model_daily):
    """Equal-weight daily portfolio metrics for one model's daily results."""
    returns = model_daily.groupby("date")["strategy_return"].mean()
    wealth = (1 + returns).cumprod()
    years = len(returns) / 252
    return {
        "annualized_return": wealth.iloc[-1] ** (1 / years) - 1 if wealth.iloc[-1] > 0 else np.nan,
        "annualized_volatility": returns.std() * np.sqrt(252),
        "sharpe_ratio": annualized_sharpe(returns),
        "sortino_ratio": annualized_sortino(returns),
        "max_drawdown": (wealth / wealth.cummax() - 1).min(),
        "average_gross_exposure": model_daily["position_gross_exposure"].mean(),
        "average_daily_turnover": model_daily["turnover"].mean(),
    }


def load_results(results_dir, run_hint):
    """Portfolio table, fold summaries, per-year portfolio Sharpe and extra CSVs."""
    portfolio_rows, fold_frames, yearly_frames, extras = [], [], [], {}
    for scenario, suffix in SCENARIOS.items():
        daily_path = results_dir / f"daily_results_{suffix}.csv"
        summary_path = results_dir / f"summary_{suffix}.csv"
        for path in (daily_path, summary_path):
            if not path.exists():
                raise FileNotFoundError(
                    f"Missing {path}. Run `{run_hint} --transaction-cost-bps {suffix[:-3]}` first."
                )
        daily = pd.read_csv(daily_path, parse_dates=["date"])
        fold_summary = pd.read_csv(summary_path)
        fold_summary["cost_scenario"] = scenario
        fold_sortino = pd.DataFrame(
            [
                {**dict(zip(FOLD_KEYS, keys)), "sortino_ratio": annualized_sortino(fold["strategy_return"])}
                for keys, fold in daily.groupby(FOLD_KEYS)
            ]
        )
        fold_frames.append(
            fold_summary.drop(columns=["sortino_ratio"], errors="ignore").merge(
                fold_sortino, on=FOLD_KEYS, validate="one_to_one"
            )
        )
        for model_name, model_daily in daily.groupby("model_name", sort=False):
            portfolio_rows.append(
                {"cost_scenario": scenario, "model_name": model_name, **portfolio_row(model_daily)}
            )
        portfolio_returns = daily.groupby(["model_name", "date"])["strategy_return"].mean().reset_index()
        yearly = (
            portfolio_returns.groupby(["model_name", portfolio_returns["date"].dt.year.rename("test_year")])
            ["strategy_return"].apply(annualized_sharpe)
            .unstack("model_name")
        )
        yearly.insert(0, "cost_scenario", scenario)
        yearly_frames.append(yearly.reset_index())
        for name in ("robustness", "selected_parameters"):
            path = results_dir / f"{name}_{suffix}.csv"
            if path.exists():
                frame = pd.read_csv(path)
                frame.insert(0, "cost_scenario", scenario)
                extras.setdefault(name, []).append(frame)
    portfolio = pd.DataFrame(portfolio_rows).set_index(["cost_scenario", "model_name"]).sort_index()
    folds = pd.concat(fold_frames, ignore_index=True)
    yearly = pd.concat(yearly_frames, ignore_index=True).set_index(["cost_scenario", "test_year"])
    extras = {name: pd.concat(frames, ignore_index=True) for name, frames in extras.items()}
    return portfolio, folds, yearly, extras


def show_results(title, results_dir, run_hint, by_pair=True):
    portfolio, folds, yearly, extras = load_results(results_dir, run_hint)
    print(f"{title}: equal-weight portfolio comparison")
    display(portfolio.round(4))
    print(f"\n{title}: mean and median yearly fold metrics")
    display(folds.groupby(["cost_scenario", "model_name"])[FOLD_METRICS].agg(["mean", "median"]).round(4))
    if by_pair:
        print(f"\n{title}: mean yearly fold Sharpe by pair")
        display(
            folds.pivot_table(index=["cost_scenario", "pair_id"], columns="model_name", values="sharpe_ratio").round(2)
        )
    print(f"\n{title}: equal-weight portfolio Sharpe by test year")
    display(yearly.round(2))
    for name, label in (("robustness", "Robustness"), ("selected_parameters", "Parameters selected for each test year")):
        if name in extras:
            frame = extras[name].drop(columns="pairs", errors="ignore")
            index = ["cost_scenario", "model_name" if name == "robustness" else "test_year"]
            print(f"\n{title}: {label.lower()}")
            display(frame.set_index(index).round(3))
    return portfolio, folds, yearly, extras

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

results_dir = Path("backtest_results") / "regression"
scenarios = {"0 bps": "0bps", "5 bps": "5bps"}
comparison_frames = []
summary_frames = []


def annualized_sortino(returns):
    downside_deviation = returns.clip(upper=0).pow(2).mean() ** 0.5
    if downside_deviation == 0:
        return np.nan
    return returns.mean() / downside_deviation * np.sqrt(252)


for scenario, suffix in scenarios.items():
    comparison_path = results_dir / f"comparison_{suffix}.csv"
    summary_path = results_dir / f"summary_{suffix}.csv"
    daily_path = results_dir / f"daily_results_{suffix}.csv"
    for result_path in (comparison_path, summary_path, daily_path):
        if not result_path.exists():
            raise FileNotFoundError(
                f"Missing {result_path}. Run regression_model.py with transaction costs "
                f"set to {suffix[:-3]} bps first."
            )

    comparison = pd.read_csv(comparison_path)
    daily = pd.read_csv(daily_path, parse_dates=["date"])
    summary = pd.read_csv(summary_path)
    comparison["cost_scenario"] = scenario
    summary["cost_scenario"] = scenario

    portfolio_rows = []
    for model_name, model_daily in daily.groupby("model_name"):
        portfolio_returns = model_daily.groupby("date")["strategy_return"].mean()
        portfolio_rows.append(
            {
                "model_name": model_name,
                "sortino_ratio": annualized_sortino(portfolio_returns),
            }
        )

    fold_keys = ["model_name", "pair_id", "fold_id", "test_year"]
    fold_sortino_rows = []
    for keys, fold in daily.groupby(fold_keys):
        fold_sortino_rows.append(
            {
                **dict(zip(fold_keys, keys)),
                "sortino_ratio": annualized_sortino(fold["strategy_return"]),
            }
        )
    summary = summary.drop(columns=["sortino_ratio"], errors="ignore").merge(
        pd.DataFrame(fold_sortino_rows), on=fold_keys, validate="one_to_one"
    )

    comparison = comparison.merge(pd.DataFrame(portfolio_rows), on="model_name")
    comparison_frames.append(comparison)
    summary_frames.append(summary)

comparison = pd.concat(comparison_frames, ignore_index=True)
summary = pd.concat(summary_frames, ignore_index=True)

print("Equal-weight portfolio comparison (Sortino calculated from daily portfolio returns):")
portfolio_columns = [
    "annualized_return",
    "annualized_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "average_gross_exposure",
    "average_daily_turnover",
]
display(
    comparison.set_index(["cost_scenario", "model_name"])[portfolio_columns]
    .sort_index()
    .round(4)
)

fold_metrics = [
    "total_return",
    "annualized_return",
    "sharpe_ratio",
    "sortino_ratio",
    "annualized_volatility",
    "max_drawdown",
    "average_gross_exposure",
    "average_daily_turnover",
    "total_transaction_cost",
]
print("\nMean and median yearly fold metrics by cost scenario and model:")
display(
    summary.groupby(["cost_scenario", "model_name"])[fold_metrics]
    .agg(["mean", "median"])
    .round(4)
)

print("\nMean yearly fold metrics by cost scenario, model, and pair:")
display(
    summary.groupby(["cost_scenario", "model_name", "pair_id"])[fold_metrics]
    .mean()
    .round(4)
)


Equal-weight portfolio comparison (Sortino calculated from daily portfolio returns):


annualized_return  annualized_volatility  \
cost_scenario model_name                                                  
0 bps         lr_regime_both              0.0081                 0.0168   
              lr_regime_gate              0.0042                 0.0133   
              lr_regime_input             0.0158                 0.0274   
              lr_regime_none              0.0202                 0.0280   
              naive_zscore                0.0347                 0.0317   
5 bps         lr_regime_both             -0.0029                 0.0168   
              lr_regime_gate             -0.0038                 0.0133   
              lr_regime_input             0.0020                 0.0273   
              lr_regime_none              0.0089                 0.0279   
              naive_zscore                0.0263                 0.0317   

                               sharpe_ratio  sortino_ratio  max_drawdown  \
cost_scenario model_name                                                   
0 bps         lr_regime_both         0.4892         0.7394       -0.0413   
              lr_regime_gate         0.3219         0.4534       -0.0403   
              lr_regime_input        0.5847         0.8509       -0.0591   
              lr_regime_none         0.7292         1.0821       -0.0585   
              naive_zscore           1.0912         1.7013       -0.0465   
5 bps         lr_regime_both        -0.1671        -0.2437       -0.0474   
              lr_regime_gate        -0.2786        -0.3809       -0.0513   
              lr_regime_input        0.0877         0.1250       -0.0627   
              lr_regime_none         0.3322         0.4840       -0.0639   
              naive_zscore           0.8363         1.2873       -0.0491   

                               average_gross_exposure  average_daily_turnover  
cost_scenario model_name                                                       
0 bps         lr_regime_both                   0.2916                  0.0873  
              lr_regime_gate                   0.2209                  0.0633  
              lr_regime_input                  0.4908                  0.1081  
              lr_regime_none                   0.4812                  0.0883  
              naive_zscore                     0.5583                  0.0645  
5 bps         lr_regime_both                   0.2916                  0.0873  
              lr_regime_gate                   0.2209                  0.0633  
              lr_regime_input                  0.4908                  0.1081  
              lr_regime_none                   0.4812                  0.0883  
              naive_zscore                     0.5583                  0.0645


Mean and median yearly fold metrics by cost scenario and model:


total_return         annualized_return          \
                                      mean  median              mean  median   
cost_scenario model_name                                                       
0 bps         lr_regime_both        0.0080  0.0101            0.0080  0.0101   
              lr_regime_gate        0.0042  0.0050            0.0042  0.0050   
              lr_regime_input       0.0158  0.0134            0.0158  0.0134   
              lr_regime_none        0.0205  0.0186            0.0205  0.0186   
              naive_zscore          0.0345  0.0351            0.0345  0.0352   
5 bps         lr_regime_both       -0.0030 -0.0008           -0.0030 -0.0008   
              lr_regime_gate       -0.0038 -0.0018           -0.0038 -0.0018   
              lr_regime_input       0.0019  0.0017            0.0019  0.0017   
              lr_regime_none        0.0091  0.0071            0.0091  0.0070   
              naive_zscore          0.0260  0.0261            0.0260  0.0261   

                              sharpe_ratio         sortino_ratio          \
                                      mean  median          mean  median   
cost_scenario model_name                                                   
0 bps         lr_regime_both        0.3532  0.4325        0.7795  0.7712   
              lr_regime_gate        0.3632  0.3341        0.8088  0.4757   
              lr_regime_input       0.3987  0.3092        0.7506  0.4692   
              lr_regime_none        0.4790  0.4615        0.9220  0.7397   
              naive_zscore          0.7193  0.6802        1.3657  1.1089   
5 bps         lr_regime_both       -0.0869 -0.0448        0.0436 -0.0546   
              lr_regime_gate       -0.0715 -0.1347        0.0679 -0.1920   
              lr_regime_input       0.0527  0.0794        0.1935  0.1163   
              lr_regime_none        0.1975  0.2148        0.4635  0.2985   
              naive_zscore          0.5220  0.5189        1.0032  0.7880   

                              annualized_volatility         max_drawdown  \
                                               mean  median         mean   
cost_scenario model_name                                                   
0 bps         lr_regime_both                 0.0317  0.0287      -0.0306   
              lr_regime_gate                 0.0236  0.0207      -0.0238   
              lr_regime_input                0.0532  0.0434      -0.0502   
              lr_regime_none                 0.0530  0.0411      -0.0495   
              naive_zscore                   0.0569  0.0506      -0.0496   
5 bps         lr_regime_both                 0.0317  0.0285      -0.0336   
              lr_regime_gate                 0.0236  0.0207      -0.0260   
              lr_regime_input                0.0531  0.0433      -0.0534   
              lr_regime_none                 0.0529  0.0411      -0.0517   
              naive_zscore                   0.0568  0.0504      -0.0504   

                                      average_gross_exposure          \
                               median                   mean  median   
cost_scenario model_name                                               
0 bps         lr_regime_both  -0.0233                 0.2920  0.2440   
              lr_regime_gate  -0.0181                 0.2212  0.1459   
              lr_regime_input -0.0391                 0.4908  0.4894   
              lr_regime_none  -0.0362                 0.4812  0.4715   
              naive_zscore    -0.0388                 0.5584  0.5574   
5 bps         lr_regime_both  -0.0266                 0.2920  0.2440   
              lr_regime_gate  -0.0202                 0.2212  0.1459   
              lr_regime_input -0.0421                 0.4908  0.4894   
              lr_regime_none  -0.0387                 0.4812  0.4715   
              naive_zscore    -0.0403                 0.5584  0.5574   

                              average_daily_turnover          \
          


Mean yearly fold metrics by cost scenario, model, and pair:


total_return  annualized_return  \
cost_scenario model_name      pair_id                                    
0 bps         lr_regime_both  all_trv       -0.0131            -0.0131   
                              bac_pnc        0.0186             0.0187   
                              hd_low         0.0111             0.0111   
                              tfc_usb        0.0198             0.0198   
                              xel_wec        0.0036             0.0036   
              lr_regime_gate  all_trv       -0.0111            -0.0112   
                              bac_pnc        0.0176             0.0177   
                              hd_low         0.0050             0.0050   
                              tfc_usb        0.0054             0.0053   
                              xel_wec        0.0041             0.0041   
              lr_regime_input all_trv        0.0002             0.0002   
                              bac_pnc        0.0497             0.0497   
                              hd_low         0.0103             0.0103   
                              tfc_usb        0.0041             0.0041   
                              xel_wec        0.0147             0.0147   
              lr_regime_none  all_trv        0.0107             0.0107   
                              bac_pnc        0.0475             0.0475   
                              hd_low         0.0300             0.0301   
                              tfc_usb       -0.0015            -0.0014   
                              xel_wec        0.0156             0.0156   
              naive_zscore    all_trv        0.0216             0.0217   
                              bac_pnc        0.0542             0.0542   
                              hd_low         0.0338             0.0339   
                              tfc_usb        0.0352             0.0351   
                              xel_wec        0.0275             0.0275   
5 bps         lr_regime_both  all_trv       -0.0234            -0.0235   
                              bac_pnc        0.0062             0.0063   
                              hd_low        -0.0006            -0.0006   
                              tfc_usb        0.0089             0.0089   
                              xel_wec       -0.0063            -0.0063   
              lr_regime_gate  all_trv       -0.0188            -0.0189   
                              bac_pnc        0.0090             0.0091   
                              hd_low        -0.0036            -0.0036   
                              tfc_usb       -0.0022            -0.0022   
                              xel_wec       -0.0033            -0.0033   
              lr_regime_input all_trv       -0.0119            -0.0119   
                              bac_pnc        0.0318             0.0317   
                              hd_low        -0.0036            -0.0037   
                              tfc_usb       -0.0090            -0.0089   
                              xel_wec        0.0024             0.0024   
              lr_regime_none  all_trv        0.0006             0.0006   
                              bac_pnc        0.0327             0.0327   
                              hd_low         0.0182             0.0182   
                              tfc_usb       -0.0111            -0.0111   
                              xel_wec        0.0052             0.0052   
              naive_zscore    all_trv        0.0143             0.0144   
                              bac_pnc        0.0442             0.0442   
                              hd_low         0.0269             0.0270   
                              tfc_usb        0.0272             0.0271   
                              xel_wec        0.0173             0.0173   

                                       sharpe_ratio  sortino_ratio  \
cost_scenario model_name      pair_id                                
0 bps         lr_regime_both  all_trv       -0.147

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

results_dir = Path("backtest_results") / "dynamic_threshold"
scenarios = {"0 bps": "0bps", "5 bps": "5bps"}
portfolio_rows = []
dynamic_fold_frames = []
fold_keys = ["model_name", "pair_id", "fold_id", "test_year"]


def annualized_sortino(returns):
    downside_deviation = returns.clip(upper=0).pow(2).mean() ** 0.5
    if downside_deviation == 0:
        return np.nan
    return returns.mean() / downside_deviation * np.sqrt(252)


for scenario, suffix in scenarios.items():
    daily_path = results_dir / f"daily_results_{suffix}.csv"
    summary_path = results_dir / f"summary_{suffix}.csv"
    for result_path in (daily_path, summary_path):
        if not result_path.exists():
            raise FileNotFoundError(
                f"Missing {result_path}. Run the dynamic-threshold model with "
                f"transaction costs set to {suffix[:-3]} bps first."
            )

    daily = pd.read_csv(daily_path, parse_dates=["date"])
    fold_summary = pd.read_csv(summary_path)
    fold_summary["cost_scenario"] = scenario
    fold_sortino = []
    for keys, fold in daily.groupby(fold_keys):
        fold_sortino.append(
            {
                **dict(zip(fold_keys, keys)),
                "sortino_ratio": annualized_sortino(fold["strategy_return"]),
            }
        )
    fold_summary = fold_summary.merge(
        pd.DataFrame(fold_sortino), on=fold_keys, validate="one_to_one"
    )
    dynamic_fold_frames.append(fold_summary)

    for model_name, model_daily in daily.groupby("model_name"):
        portfolio_returns = model_daily.groupby("date")["strategy_return"].mean()
        daily_volatility = portfolio_returns.std()
        wealth = (1 + portfolio_returns).cumprod()
        years = len(portfolio_returns) / 252
        portfolio_rows.append(
            {
                "cost_scenario": scenario,
                "model_name": model_name,
                "annualized_return": (
                    wealth.iloc[-1] ** (1 / years) - 1 if wealth.iloc[-1] > 0 else np.nan
                ),
                "annualized_volatility": daily_volatility * np.sqrt(252),
                "sharpe_ratio": (
                    portfolio_returns.mean() / daily_volatility * np.sqrt(252)
                    if daily_volatility > 0
                    else np.nan
                ),
                "sortino_ratio": annualized_sortino(portfolio_returns),
                "max_drawdown": (wealth / wealth.cummax() - 1).min(),
                "average_gross_exposure": model_daily[
                    "position_gross_exposure"
                ].mean(),
                "average_daily_turnover": model_daily["turnover"].mean(),
            }
        )

print("Dynamic-threshold equal-weight portfolio comparison:")
display(
    pd.DataFrame(portfolio_rows)
    .set_index(["cost_scenario", "model_name"])
    .sort_index()
    .round(4)
)

dynamic_fold_summary = pd.concat(dynamic_fold_frames, ignore_index=True)
fold_metrics = [
    "total_return",
    "annualized_return",
    "sharpe_ratio",
    "sortino_ratio",
    "annualized_volatility",
    "max_drawdown",
    "average_gross_exposure",
    "average_daily_turnover",
    "total_transaction_cost",
]
print("\nMean yearly fold metrics by cost scenario and pair:")
display(
    dynamic_fold_summary.groupby(["cost_scenario", "pair_id"])[fold_metrics]
    .mean()
    .round(4)
)


Dynamic-threshold equal-weight portfolio comparison:


,,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,max_drawdown,average_gross_exposure,average_daily_turnover
cost_scenario,model_name,,,,,,,
0 bps,sharpe_optimized_threshold,0.0256,0.0248,1.0316,1.6201,-0.0471,0.3064,0.0353
5 bps,sharpe_optimized_threshold,0.0215,0.0251,0.8596,1.3394,-0.0490,0.2987,0.0337



Mean yearly fold metrics by cost scenario and pair:


total_return  annualized_return  sharpe_ratio  \
cost_scenario pair_id                                                  
0 bps         all_trv        0.0199             0.0199        0.4613   
              bac_pnc        0.0247             0.0248        0.9594   
              hd_low         0.0290             0.0290        0.8410   
              tfc_usb        0.0307             0.0306        0.7955   
              xel_wec        0.0218             0.0218        0.9787   
5 bps         all_trv        0.0116             0.0116        0.2527   
              bac_pnc        0.0303             0.0303        0.9040   
              hd_low         0.0215             0.0215        0.7019   
              tfc_usb        0.0271             0.0271        0.7265   
              xel_wec        0.0150             0.0150        0.7505   

                       sortino_ratio  annualized_volatility  max_drawdown  \
cost_scenario pair_id                                                       
0 bps         all_trv         1.0058                 0.0458       -0.0437   
              bac_pnc         1.7517                 0.0444       -0.0378   
              hd_low          1.9076                 0.0374       -0.0224   
              tfc_usb         1.4177                 0.0472       -0.0388   
              xel_wec         3.3020                 0.0235       -0.0159   
5 bps         all_trv         0.5851                 0.0464       -0.0467   
              bac_pnc         1.8052                 0.0473       -0.0388   
              hd_low          1.5441                 0.0342       -0.0211   
              tfc_usb         1.3840                 0.0463       -0.0381   
              xel_wec         1.5681                 0.0226       -0.0176   

                       average_gross_exposure  average_daily_turnover  \
cost_scenario pair_id                                                   
0 bps         all_trv                  0.3264                  0.0279   
              bac_pnc                  0.2764                  0.0359   
              hd_low                   0.2584                  0.0324   
              tfc_usb                  0.3614                  0.0356   
              xel_wec                  0.3097                  0.0448   
5 bps         all_trv                  0.3274                  0.0276   
              bac_pnc                  0.3024                  0.0412   
              hd_low                   0.2335                  0.0287   
              tfc_usb                  0.3320                  0.0315   
              xel_wec                  0.2986                  0.0392   

                       total_transaction_cost  
cost_scenario pair_id                          
0 bps         all_trv                  0.0000  
              bac_pnc                  0.0000  
              hd_low                   0.0000  
              tfc_usb                  0.0000  
              xel_wec                  0.0000  
5 bps         all_trv                  0.0035  
              bac_pnc                  0.0052  
              hd_low                   0.0036  
              tfc_usb                  0.0040  
              xel_wec                  0.0049

## Regime regression with EWMA z-score

`improved_models/regime_regression_model.py`: the regression model with an EWMA z-score, the VIX tercile regime and a VIX top/bottom-10% state as linear inputs (no gating). Ablations: `regime_lr_static_z` keeps the static z-score, `regime_lr_ewma_z_no_extreme` drops the VIX-extreme state.

In [4]:
regime_results = show_results(
    "Regime regression",
    Path("backtest_results") / "regime_regression",
    "python -m improved_models.regime_regression_model",
)

Regime regression: equal-weight portfolio comparison


annualized_return  \
cost_scenario model_name                                       
0 bps         lr_regime_input                         0.0158   
              naive_zscore                            0.0347   
              regime_lr_ewma_z                        0.0037   
              regime_lr_ewma_z_no_extreme             0.0052   
              regime_lr_static_z                      0.0101   
5 bps         lr_regime_input                         0.0020   
              naive_zscore                            0.0263   
              regime_lr_ewma_z                       -0.0145   
              regime_lr_ewma_z_no_extreme            -0.0101   
              regime_lr_static_z                     -0.0065   

                                           annualized_volatility  \
cost_scenario model_name                                           
0 bps         lr_regime_input                             0.0274   
              naive_zscore                                0.0317   
              regime_lr_ewma_z                            0.0271   
              regime_lr_ewma_z_no_extreme                 0.0270   
              regime_lr_static_z                          0.0301   
5 bps         lr_regime_input                             0.0273   
              naive_zscore                                0.0317   
              regime_lr_ewma_z                            0.0270   
              regime_lr_ewma_z_no_extreme                 0.0270   
              regime_lr_static_z                          0.0301   

                                           sharpe_ratio  sortino_ratio  \
cost_scenario model_name                                                 
0 bps         lr_regime_input                    0.5847         0.8509   
              naive_zscore                       1.0912         1.7013   
              regime_lr_ewma_z                   0.1504         0.2170   
              regime_lr_ewma_z_no_extreme        0.2064         0.3025   
              regime_lr_static_z                 0.3492         0.4919   
5 bps         lr_regime_input                    0.0877         0.1250   
              naive_zscore                       0.8363         1.2873   
              regime_lr_ewma_z                  -0.5274        -0.7371   
              regime_lr_ewma_z_no_extreme       -0.3637        -0.5185   
              regime_lr_static_z                -0.2025        -0.2793   

                                           max_drawdown  \
cost_scenario model_name                                  
0 bps         lr_regime_input                   -0.0591   
              naive_zscore                      -0.0465   
              regime_lr_ewma_z                  -0.0725   
              regime_lr_ewma_z_no_extreme       -0.0675   
              regime_lr_static_z                -0.0538   
5 bps         lr_regime_input                   -0.0627   
              naive_zscore                      -0.0491   
              regime_lr_ewma_z                  -0.1772   
              regime_lr_ewma_z_no_extreme       -0.1346   
              regime_lr_static_z                -0.0933   

                                           average_gross_exposure  \
cost_scenario model_name                                            
0 bps         lr_regime_input                              0.4908   
              naive_zscore                                 0.5583   
              regime_lr_ewma_z                             0.4833   
              regime_lr_ewma_z_no_extreme                  0.4781   
              regime_lr_static_z                           0.4982   
5 bps         lr_regime_input                              0.4908   
              naive_zscore                                 0.5583   
              regime_lr_ewma_z                             0.4833   
              regime_lr_ewma_z_no_extreme                  0.4781   
              regime_lr_static_z                           0.4982   

                              


Regime regression: mean and median yearly fold metrics


total_return          \
                                                  mean  median   
cost_scenario model_name                                         
0 bps         lr_regime_input                   0.0158  0.0134   
              naive_zscore                      0.0345  0.0351   
              regime_lr_ewma_z                  0.0038  0.0091   
              regime_lr_ewma_z_no_extreme       0.0049  0.0166   
              regime_lr_static_z                0.0108  0.0133   
5 bps         lr_regime_input                   0.0019  0.0017   
              naive_zscore                      0.0260  0.0261   
              regime_lr_ewma_z                 -0.0144 -0.0072   
              regime_lr_ewma_z_no_extreme      -0.0104  0.0013   
              regime_lr_static_z               -0.0058  0.0007   

                                          annualized_return          \
                                                       mean  median   
cost_scenario model_name                                              
0 bps         lr_regime_input                        0.0158  0.0134   
              naive_zscore                           0.0345  0.0352   
              regime_lr_ewma_z                       0.0038  0.0092   
              regime_lr_ewma_z_no_extreme            0.0050  0.0166   
              regime_lr_static_z                     0.0108  0.0133   
5 bps         lr_regime_input                        0.0019  0.0017   
              naive_zscore                           0.0260  0.0261   
              regime_lr_ewma_z                      -0.0144 -0.0072   
              regime_lr_ewma_z_no_extreme           -0.0104  0.0013   
              regime_lr_static_z                    -0.0059  0.0007   

                                          sharpe_ratio         sortino_ratio  \
                                                  mean  median          mean   
cost_scenario model_name                                                       
0 bps         lr_regime_input                   0.3987  0.3092        0.7506   
              naive_zscore                      0.7193  0.6802        1.3657   
              regime_lr_ewma_z                  0.2106  0.2167        0.4755   
              regime_lr_ewma_z_no_extreme       0.2312  0.4474        0.4855   
              regime_lr_static_z                0.3396  0.3634        0.7271   
5 bps         lr_regime_input                   0.0527  0.0794        0.1935   
              naive_zscore                      0.5220  0.5189        1.0032   
              regime_lr_ewma_z                 -0.2525 -0.1022       -0.2326   
              regime_lr_ewma_z_no_extreme      -0.1688  0.0679       -0.1239   
              regime_lr_static_z               -0.0741  0.0383        0.0547   

                                                  annualized_volatility  \
                                           median                  mean   
cost_scenario model_name                                                  
0 bps         lr_regime_input              0.4692                0.0532   
              naive_zscore                 1.1089                0.0569   
              regime_lr_ewma_z             0.3273                0.0529   
              regime_lr_ewma_z_no_extreme  0.6828                0.0527   
              regime_lr_static_z           0.5362                0.0541   
5 bps         lr_regime_input              0.1163                0.0531   
              naive_zscore                 0.7880                0.0568   
              regime_lr_ewma_z            -0.1744                0.0528   
              regime_lr_ewma_z_no_extreme  0.1003                0.0526   
              regime_lr_static_z           0.0573                0.0540   

                                                  max_drawdown          \
                                           median         mean  median   
cost_scenario model_name                                                 
0 bps         lr_reg


Regime regression: mean yearly fold Sharpe by pair


model_name             lr_regime_input  naive_zscore  regime_lr_ewma_z  \
cost_scenario pair_id                                                    
0 bps         all_trv             0.07          0.39              0.07   
              bac_pnc             0.77          0.95              0.30   
              hd_low              0.12          0.52              0.10   
              tfc_usb             0.55          0.78              0.02   
              xel_wec             0.49          0.97              0.56   
5 bps         all_trv            -0.17          0.26             -0.26   
              bac_pnc             0.44          0.77             -0.10   
              hd_low             -0.16          0.40             -0.33   
              tfc_usb             0.21          0.59             -0.45   
              xel_wec            -0.06          0.58             -0.12   

model_name             regime_lr_ewma_z_no_extreme  regime_lr_static_z  
cost_scenario pair_id                                                   
0 bps         all_trv                         0.09                0.06  
              bac_pnc                         0.39                0.70  
              hd_low                          0.01                0.07  
              tfc_usb                         0.11                0.47  
              xel_wec                         0.56                0.41  
5 bps         all_trv                        -0.18               -0.25  
              bac_pnc                         0.04                0.31  
              hd_low                         -0.34               -0.27  
              tfc_usb                        -0.27                0.01  
              xel_wec                        -0.09               -0.17


Regime regression: equal-weight portfolio Sharpe by test year


model_name               lr_regime_input  naive_zscore  regime_lr_ewma_z  \
cost_scenario test_year                                                    
0 bps         2014                  0.54          0.82              1.53   
              2015                  0.86          0.92             -0.14   
              2016                 -0.05          0.37              0.24   
              2017                 -0.62         -0.64             -2.09   
              2018                  0.80          1.86             -0.47   
              2019                  1.46          0.82              0.92   
              2020                  0.94          2.04             -0.30   
              2021                 -0.07          0.72             -0.44   
              2022                  2.32          3.99              1.35   
              2023                  0.11          0.73              0.67   
5 bps         2014                 -0.20          0.42              0.27   
              2015                  0.13          0.49             -1.08   
              2016                 -0.40          0.21             -0.44   
              2017                 -1.24         -0.88             -3.09   
              2018                  0.12          1.53             -1.28   
              2019                  0.75          0.55              0.12   
              2020                  0.66          1.84             -0.68   
              2021                 -0.73          0.42             -1.26   
              2022                  1.62          3.47              0.60   
              2023                 -0.30          0.51              0.18   

model_name               regime_lr_ewma_z_no_extreme  regime_lr_static_z  
cost_scenario test_year                                                   
0 bps         2014                              1.25                0.46  
              2015                              0.14                0.68  
              2016                             -0.13                0.49  
              2017                             -1.55               -1.64  
              2018                             -0.38                0.87  
              2019                              1.13                1.55  
              2020                             -0.27                0.25  
              2021                             -0.31               -0.32  
              2022                              1.79                2.20  
              2023                              0.53               -0.26  
5 bps         2014                              0.17               -0.45  
              2015                             -0.65               -0.24  
              2016                             -0.61               -0.08  
              2017                             -2.33               -2.42  
              2018                             -1.08                0.11  
              2019                              0.48                0.74  
              2020                             -0.62                0.00  
              2021                             -1.09               -1.06  
              2022                              1.12                1.40  
              2023                              0.08               -0.70

## Vol-scaled naive model

`improved_models/vol_scaled_naive_model.py`: naive sizing `clip(-z / 2, -1, 1)` on an EWMA z-score, multiplied by a VIX-percentile size overlay, with a no-trade band that is wider when VIX is low. `vix_scale`, `band` and `band_vix_scale` are chosen each test year on the training window only, pooled across pairs. `naive_ewma_z` isolates the z-score change (no overlay, no band).

Robustness columns: `sharpe_ex_2020_2022` drops the two crisis years; `ann_return_at_baseline_vol` rescales each model to the naive model's volatility; the `gap_vs_baseline` percentiles and `prob_beats_baseline` come from a bootstrap that resamples whole years.

### Original screened pairs (pair selection overlaps the test period)

In [5]:
original_results = show_results(
    "Vol-scaled naive, original pairs",
    Path("backtest_results") / "vol_scaled_naive" / "original_pairs",
    "python -m improved_models.vol_scaled_naive_model --input data/screened_pair_prices.csv "
    "--selection none --output-dir backtest_results/vol_scaled_naive/original_pairs",
)

Vol-scaled naive, original pairs: equal-weight portfolio comparison


annualized_return  annualized_volatility  \
cost_scenario model_name                                                   
0 bps         naive_ewma_z                 0.0293                 0.0256   
              naive_zscore                 0.0347                 0.0317   
              vol_scaled_naive             0.0289                 0.0267   
5 bps         naive_ewma_z                 0.0199                 0.0255   
              naive_zscore                 0.0263                 0.0317   
              vol_scaled_naive             0.0250                 0.0270   

                                sharpe_ratio  sortino_ratio  max_drawdown  \
cost_scenario model_name                                                    
0 bps         naive_ewma_z            1.1426         1.8252       -0.0394   
              naive_zscore            1.0912         1.7013       -0.0465   
              vol_scaled_naive        1.0773         1.7216       -0.0407   
5 bps         naive_ewma_z            0.7861         1.2318       -0.0423   
              naive_zscore            0.8363         1.2873       -0.0491   
              vol_scaled_naive        0.9300         1.4706       -0.0419   

                                average_gross_exposure  average_daily_turnover  
cost_scenario model_name                                                        
0 bps         naive_ewma_z                      0.4333                  0.0727  
              naive_zscore                      0.5583                  0.0645  
              vol_scaled_naive                  0.4382                  0.0397  
5 bps         naive_ewma_z                      0.4333                  0.0727  
              naive_zscore                      0.5583                  0.0645  
              vol_scaled_naive                  0.4493                  0.0303


Vol-scaled naive, original pairs: mean and median yearly fold metrics


total_return         annualized_return          \
                                       mean  median              mean  median   
cost_scenario model_name                                                        
0 bps         naive_ewma_z           0.0290  0.0273            0.0291  0.0274   
              naive_zscore           0.0345  0.0351            0.0345  0.0352   
              vol_scaled_naive       0.0287  0.0232            0.0287  0.0232   
5 bps         naive_ewma_z           0.0196  0.0182            0.0196  0.0183   
              naive_zscore           0.0260  0.0261            0.0260  0.0261   
              vol_scaled_naive       0.0248  0.0176            0.0248  0.0176   

                               sharpe_ratio         sortino_ratio          \
                                       mean  median          mean  median   
cost_scenario model_name                                                    
0 bps         naive_ewma_z           0.7165  0.7916        1.3373  1.3936   
              naive_zscore           0.7193  0.6802        1.3657  1.1089   
              vol_scaled_naive       0.6458  0.7452        1.2037  1.2652   
5 bps         naive_ewma_z           0.4527  0.5977        0.8669  1.0131   
              naive_zscore           0.5220  0.5189        1.0032  0.7880   
              vol_scaled_naive       0.5321  0.5440        0.9843  0.7982   

                               annualized_volatility         max_drawdown  \
                                                mean  median         mean   
cost_scenario model_name                                                    
0 bps         naive_ewma_z                    0.0461  0.0391      -0.0398   
              naive_zscore                    0.0569  0.0506      -0.0496   
              vol_scaled_naive                0.0470  0.0413      -0.0403   
5 bps         naive_ewma_z                    0.0460  0.0391      -0.0409   
              naive_zscore                    0.0568  0.0504      -0.0504   
              vol_scaled_naive                0.0477  0.0413      -0.0415   

                                       average_gross_exposure          \
                                median                   mean  median   
cost_scenario model_name                                                
0 bps         naive_ewma_z     -0.0281                 0.4334  0.4378   
              naive_zscore     -0.0388                 0.5584  0.5574   
              vol_scaled_naive -0.0292                 0.4382  0.4433   
5 bps         naive_ewma_z     -0.0315                 0.4334  0.4378   
              naive_zscore     -0.0403                 0.5584  0.5574   
              vol_scaled_naive -0.0331                 0.4493  0.4567   

                               average_daily_turnover          \
                                                 mean  median   
cost_scenario model_name                                        
0 bps         naive_ewma_z                     0.0727  0.0718   
              naive_zscore                     0.0645  0.0607   
              vol_scaled_naive                 0.0397  0.0332   
5 bps         naive_ewma_z                     0.0727  0.0718   
              naive_zscore                     0.0645  0.0607   
              vol_scaled_naive                 0.0303  0.0240   

                               total_transaction_cost          
                                                 mean  median  
cost_scenario model_name                                       
0 bps         naive_ewma_z                     0.0000  0.0000  
              naive_zscore                     0.0000  0.0000  
              vol_scaled_naive                 0.0000  0.0000  
5 bps         naive_ewma_z                     0.0091  0.0090  
              naive_zscore                     0.0081  0.0076  
              vol_scaled_naive                 0.0038  0.0030


Vol-scaled naive, original pairs: mean yearly fold Sharpe by pair


model_name             naive_ewma_z  naive_zscore  vol_scaled_naive
cost_scenario pair_id                                              
0 bps         all_trv          0.28          0.39              0.27
              bac_pnc          0.88          0.95              0.79
              hd_low           0.67          0.52              0.54
              tfc_usb          0.68          0.78              0.64
              xel_wec          1.07          0.97              0.99
5 bps         all_trv          0.12          0.26              0.12
              bac_pnc          0.67          0.77              0.80
              hd_low           0.49          0.40              0.41
              tfc_usb          0.43          0.59              0.57
              xel_wec          0.56          0.58              0.76

... (2 outputs left out)


Vol-scaled naive, original pairs: robustness


sharpe  sharpe_ex_2020_2022  \
cost_scenario model_name                                      
0 bps         naive_ewma_z       1.143                0.711   
              naive_zscore       1.091                0.645   
              vol_scaled_naive   1.077                0.636   
5 bps         naive_ewma_z       0.786                0.321   
              naive_zscore       0.836                0.382   
              vol_scaled_naive   0.930                0.489   

                                ann_return_at_baseline_vol  sharpe_p05  \
cost_scenario model_name                                                 
0 bps         naive_ewma_z                           0.036       0.673   
              naive_zscore                           0.035       0.592   
              vol_scaled_naive                       0.034       0.582   
5 bps         naive_ewma_z                           0.025       0.286   
              naive_zscore                           0.026       0.329   
              vol_scaled_naive                       0.029       0.438   

                                sharpe_p95  gap_vs_baseline_p05  \
cost_scenario model_name                                          
0 bps         naive_ewma_z           1.621               -0.051   
              naive_zscore           1.619                0.000   
              vol_scaled_naive       1.549               -0.195   
5 bps         naive_ewma_z           1.245               -0.155   
              naive_zscore           1.333                0.000   
              vol_scaled_naive       1.370               -0.080   

                                gap_vs_baseline_p95  prob_beats_baseline  
cost_scenario model_name                                                  
0 bps         naive_ewma_z                    0.143                0.789  
              naive_zscore                    0.000                0.000  
              vol_scaled_naive                0.113                0.430  
5 bps         naive_ewma_z                    0.031                0.146  
              naive_zscore                    0.000                0.000  
              vol_scaled_naive                0.241                0.830


Vol-scaled naive, original pairs: parameters selected for each test year


vix_scale  band  band_vix_scale
cost_scenario test_year                                 
0 bps         2014            0.00   0.0             0.0
              2015            0.00   0.0             0.0
              2016            0.25   0.2             0.0
              2017            0.25   0.2             0.0
              2018            0.25   0.3             0.5
              2019            0.50   0.4             0.5
              2020            0.50   0.2             0.0
              2021            0.00   0.4             0.0
              2022            0.00   0.4             0.0
              2023            0.00   0.3             0.5
5 bps         2014            0.00   0.2             0.0
              2015            0.25   0.2             0.0
              2016            0.00   0.4             0.5
              2017            0.00   0.4             0.5
              2018            0.25   0.3             0.5
              2019            0.50   0.4             0.5
              2020            0.50   0.4             0.5
              2021            0.00   0.4             0.0
              2022            0.00   0.4             0.0
              2023            0.00   0.3             0.5

### Point-in-time pairs (no pair-selection look-ahead)

Pairs are re-screened before each test year using only the prior ten years (`build_point_in_time_dataset.py`), so the set of traded pairs changes year to year. This is the fairer test of whether the model beats the naive baseline.

In [6]:
selection_path = Path("data") / "pit_pair_selection.csv"
if selection_path.exists():
    selection = pd.read_csv(selection_path)
    print("Pairs selected for each test year:")
    display(selection.groupby("test_year")["pair_id"].agg(", ".join).to_frame("pairs"))

pit_results = show_results(
    "Vol-scaled naive, point-in-time pairs",
    Path("backtest_results") / "vol_scaled_naive",
    "python build_point_in_time_dataset.py && python -m improved_models.vol_scaled_naive_model",
    by_pair=False,
)

Pairs selected for each test year:


,pairs
test_year,
2014,"xel_wec, pnc_usb, trv_cb"
2015,"wec_xel, pnc_usb, cb_trv, pg_kmb, jpm_wfc"
2016,"xel_wec, pnc_usb, slb_cop, cb_trv, jpm_wfc"
2017,"wec_xel, tfc_pnc, pg_pep, csx_nsc, trv_cb"
2018,"tfc_pnc, pg_pep, trv_cb, cl_kmb, oxy_slb"
2019,"c_gs, trv_all, wec_aep, maa_udr, cl_kmb"
2020,"c_gs, wec_aep, noc_lmt, tfc_pnc, cl_kmb"
2021,"ess_udr, lmt_noc, trv_all, ko_pep, tfc_pnc"
2022,"ess_udr, bac_pnc, mdlz_pep, all_cb, wec_xel"


Vol-scaled naive, point-in-time pairs: equal-weight portfolio comparison


annualized_return  annualized_volatility  \
cost_scenario model_name                                                   
0 bps         naive_ewma_z                 0.0166                 0.0310   
              naive_zscore                 0.0167                 0.0360   
              vol_scaled_naive             0.0171                 0.0299   
5 bps         naive_ewma_z                 0.0075                 0.0310   
              naive_zscore                 0.0091                 0.0360   
              vol_scaled_naive             0.0129                 0.0299   

                                sharpe_ratio  sortino_ratio  max_drawdown  \
cost_scenario model_name                                                    
0 bps         naive_ewma_z            0.5469         0.8151       -0.0573   
              naive_zscore            0.4789         0.7095       -0.0537   
              vol_scaled_naive        0.5805         0.8820       -0.0546   
5 bps         naive_ewma_z            0.2584         0.3802       -0.0772   
              naive_zscore            0.2697         0.3961       -0.0612   
              vol_scaled_naive        0.4437         0.6690       -0.0552   

                                average_gross_exposure  average_daily_turnover  
cost_scenario model_name                                                        
0 bps         naive_ewma_z                      0.4773                  0.0713  
              naive_zscore                      0.6169                  0.0601  
              vol_scaled_naive                  0.4537                  0.0327  
5 bps         naive_ewma_z                      0.4773                  0.0713  
              naive_zscore                      0.6169                  0.0601  
              vol_scaled_naive                  0.4561                  0.0302


Vol-scaled naive, point-in-time pairs: mean and median yearly fold metrics


total_return         annualized_return          \
                                       mean  median              mean  median   
cost_scenario model_name                                                        
0 bps         naive_ewma_z           0.0163  0.0125            0.0163  0.0125   
              naive_zscore           0.0162  0.0128            0.0162  0.0128   
              vol_scaled_naive       0.0171  0.0167            0.0171  0.0168   
5 bps         naive_ewma_z           0.0071  0.0034            0.0072  0.0034   
              naive_zscore           0.0085  0.0065            0.0085  0.0065   
              vol_scaled_naive       0.0128  0.0105            0.0128  0.0105   

                               sharpe_ratio         sortino_ratio          \
                                       mean  median          mean  median   
cost_scenario model_name                                                    
0 bps         naive_ewma_z           0.4610  0.3808        0.9110  0.5811   
              naive_zscore           0.4551  0.3619        0.8835  0.5080   
              vol_scaled_naive       0.4528  0.4636        0.8952  0.6789   
5 bps         naive_ewma_z           0.2084  0.1296        0.4775  0.2035   
              naive_zscore           0.2650  0.2231        0.5490  0.3168   
              vol_scaled_naive       0.3401  0.2930        0.6748  0.4636   

                               annualized_volatility         max_drawdown  \
                                                mean  median         mean   
cost_scenario model_name                                                    
0 bps         naive_ewma_z                    0.0492  0.0411      -0.0483   
              naive_zscore                    0.0589  0.0510      -0.0557   
              vol_scaled_naive                0.0471  0.0376      -0.0452   
5 bps         naive_ewma_z                    0.0491  0.0411      -0.0495   
              naive_zscore                    0.0588  0.0508      -0.0565   
              vol_scaled_naive                0.0470  0.0376      -0.0455   

                                       average_gross_exposure          \
                                median                   mean  median   
cost_scenario model_name                                                
0 bps         naive_ewma_z     -0.0382                 0.4773  0.4943   
              naive_zscore     -0.0472                 0.6168  0.6350   
              vol_scaled_naive -0.0382                 0.4536  0.4618   
5 bps         naive_ewma_z     -0.0396                 0.4773  0.4943   
              naive_zscore     -0.0482                 0.6168  0.6350   
              vol_scaled_naive -0.0379                 0.4560  0.4597   

                               average_daily_turnover          \
                                                 mean  median   
cost_scenario model_name                                        
0 bps         naive_ewma_z                     0.0713  0.0673   
              naive_zscore                     0.0601  0.0575   
              vol_scaled_naive                 0.0327  0.0272   
5 bps         naive_ewma_z                     0.0713  0.0673   
              naive_zscore                     0.0601  0.0575   
              vol_scaled_naive                 0.0302  0.0227   

                               total_transaction_cost          
                                                 mean  median  
cost_scenario model_name                                       
0 bps         naive_ewma_z                     0.0000  0.0000  
              naive_zscore                     0.0000  0.0000  
              vol_scaled_naive                 0.0000  0.0000  
5 bps         naive_ewma_z                     0.0090  0.0084  
              naive_zscore                     0.0076  0.0072  
              vol_scaled_naive                 0.0038  0.0028

... (2 outputs left out)


Vol-scaled naive, point-in-time pairs: robustness


sharpe  sharpe_ex_2020_2022  \
cost_scenario model_name                                      
0 bps         naive_ewma_z       0.547                0.431   
              naive_zscore       0.479                0.393   
              vol_scaled_naive   0.580                0.513   
5 bps         naive_ewma_z       0.258               -0.009   
              naive_zscore       0.270                0.095   
              vol_scaled_naive   0.444                0.291   

                                ann_return_at_baseline_vol  sharpe_p05  \
cost_scenario model_name                                                 
0 bps         naive_ewma_z                           0.020       0.293   
              naive_zscore                           0.017       0.227   
              vol_scaled_naive                       0.021       0.363   
5 bps         naive_ewma_z                           0.009      -0.087   
              naive_zscore                           0.010      -0.013   
              vol_scaled_naive                       0.016       0.210   

                                sharpe_p95  gap_vs_baseline_p05  \
cost_scenario model_name                                          
0 bps         naive_ewma_z           0.989               -0.105   
              naive_zscore           0.931                0.000   
              vol_scaled_naive       1.061                0.001   
5 bps         naive_ewma_z           0.555               -0.236   
              naive_zscore           0.606                0.000   
              vol_scaled_naive       0.826                0.086   

                                gap_vs_baseline_p95  prob_beats_baseline  
cost_scenario model_name                                                  
0 bps         naive_ewma_z                    0.208                0.782  
              naive_zscore                    0.000                0.000  
              vol_scaled_naive                0.245                0.952  
5 bps         naive_ewma_z                    0.119                0.464  
              naive_zscore                    0.000                0.000  
              vol_scaled_naive                0.326                1.000


Vol-scaled naive, point-in-time pairs: parameters selected for each test year


vix_scale  band  band_vix_scale
cost_scenario test_year                                 
0 bps         2014             0.0   0.4             0.5
              2015             0.5   0.2             0.0
              2016             0.5   0.2             0.0
              2017             0.5   0.3             0.5
              2018             0.5   0.2             0.0
              2019             0.5   0.2             0.5
              2020             0.0   0.4             0.0
              2021             0.5   0.4             0.0
              2022             0.0   0.4             0.5
              2023             0.5   0.4             0.0
5 bps         2014             0.0   0.4             0.5
              2015             0.5   0.2             0.0
              2016             0.5   0.3             0.5
              2017             0.5   0.3             0.5
              2018             0.5   0.2             0.0
              2019             0.0   0.4             0.0
              2020             0.0   0.4             0.0
              2021             0.5   0.4             0.0
              2022             0.0   0.4             0.5
              2023             0.5   0.4             0.0

## Entry-state model (point-in-time pairs)

`improved_models/entry_state_model.py`: EWMA z-score trades that open at |z| > 1 and close when z crosses zero. Each trade's size is set when it opens: full size if VIX is above its trailing 75th percentile, or the spread's 20-day volatility exceeds 1.3x its 120-day volatility. Otherwise the trade is "calm drift" and uses `calm_size` (1, 0.5 or 0). `no_upgrade` variants never resize a calm trade later. `ewma_z_band` is the EWMA z-score plus a fixed 0.2 band, with no volatility conditioning.

In [7]:
entry_results = show_results(
    "Entry-state model, point-in-time pairs",
    Path("backtest_results") / "entry_state",
    "python -m improved_models.entry_state_model",
    by_pair=False,
)

# Per-trade P&L by state at entry (no-cost run, calm_size=1 so every trade is held at full size).
trade_daily = pd.read_csv(
    Path("backtest_results") / "entry_state" / "daily_results_0bps.csv", parse_dates=["date"], low_memory=False
)
trade_daily = trade_daily[trade_daily["model_name"] == "entry_state_calm_1"].sort_values(["pair_id", "fold_id", "date"])
for column in ("trade_entry", "trade_full_size"):
    trade_daily[column] = trade_daily[column].astype(str) == "True"
trade_daily["trade_id"] = trade_daily.groupby(["pair_id", "fold_id"])["trade_entry"].cumsum()
trade_daily["next_return"] = trade_daily.groupby(["pair_id", "fold_id"])["gross_strategy_return"].shift(-1)
open_days = trade_daily[trade_daily["trade_side"].astype(float) != 0]
trades = open_days.groupby(["pair_id", "fold_id", "trade_id"]).agg(
    test_year=("test_year", "first"),
    favourable_at_entry=("trade_full_size", "first"),
    full_at_exit=("trade_full_size", "last"),
    days=("date", "size"),
    pnl=("next_return", "sum"),
)
trades["state"] = np.select(
    [trades["favourable_at_entry"], trades["full_at_exit"]],
    ["favourable at entry", "calm, upgraded later"],
    "calm throughout",
)


def trade_table(frame):
    grouped = frame.groupby("state")["pnl"]
    return pd.DataFrame(
        {
            "trades": grouped.size(),
            "avg_days": frame.groupby("state")["days"].mean().round(0),
            "avg_pnl_bps": (grouped.mean() * 1e4).round(1),
            "median_pnl_bps": (grouped.median() * 1e4).round(1),
            "win_rate": grouped.apply(lambda pnl: (pnl > 0).mean()).round(2),
        }
    )


print("\nPer-trade gross P&L by state at entry (all years):")
display(trade_table(trades))
print("\nExcluding 2020 and 2022:")
display(trade_table(trades[~trades["test_year"].isin([2020, 2022])]))

C:\Users\bened\AppData\Local\Temp\ipykernel_20572\1413721630.py:60: DtypeWarning: Columns (36,41,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  daily = pd.read_csv(daily_path, parse_dates=["date"])


C:\Users\bened\AppData\Local\Temp\ipykernel_20572\1413721630.py:60: DtypeWarning: Columns (36,41,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  daily = pd.read_csv(daily_path, parse_dates=["date"])


Entry-state model, point-in-time pairs: equal-weight portfolio comparison


annualized_return  \
cost_scenario model_name                                           
0 bps         entry_state_calm_0                          0.0144   
              entry_state_calm_0.5                        0.0141   
              entry_state_calm_0.5_no_upgrade             0.0117   
              entry_state_calm_0_no_upgrade               0.0120   
              entry_state_calm_1                          0.0155   
              ewma_z_band                                 0.0170   
              naive_zscore                                0.0167   
5 bps         entry_state_calm_0                          0.0114   
              entry_state_calm_0.5                        0.0109   
              entry_state_calm_0.5_no_upgrade             0.0094   
              entry_state_calm_0_no_upgrade               0.0104   
              entry_state_calm_1                          0.0118   
              ewma_z_band                                 0.0124   
              naive_zscore                                0.0091   

                                               annualized_volatility  \
cost_scenario model_name                                               
0 bps         entry_state_calm_0                              0.0286   
              entry_state_calm_0.5                            0.0289   
              entry_state_calm_0.5_no_upgrade                 0.0272   
              entry_state_calm_0_no_upgrade                   0.0262   
              entry_state_calm_1                              0.0295   
              ewma_z_band                                     0.0302   
              naive_zscore                                    0.0360   
5 bps         entry_state_calm_0                              0.0286   
              entry_state_calm_0.5                            0.0289   
              entry_state_calm_0.5_no_upgrade                 0.0271   
              entry_state_calm_0_no_upgrade                   0.0261   
              entry_state_calm_1                              0.0295   
              ewma_z_band                                     0.0301   
              naive_zscore                                    0.0360   

                                               sharpe_ratio  sortino_ratio  \
cost_scenario model_name                                                     
0 bps         entry_state_calm_0                     0.5149         0.7752   
              entry_state_calm_0.5                   0.4973         0.7457   
              entry_state_calm_0.5_no_upgrade        0.4418         0.6566   
              entry_state_calm_0_no_upgrade          0.4699         0.6995   
              entry_state_calm_1                     0.5348         0.8010   
              ewma_z_band                            0.5728         0.8569   
              naive_zscore                           0.4789         0.7095   
5 bps         entry_state_calm_0                     0.4110         0.6150   
              entry_state_calm_0.5                   0.3890         0.5795   
              entry_state_calm_0.5_no_upgrade        0.3571         0.5280   
              entry_state_calm_0_no_upgrade          0.4070         0.6030   
              entry_state_calm_1                     0.4122         0.6130   
              ewma_z_band                            0.4244         0.6300   
              naive_zscore                           0.2697         0.3961   

                                               max_drawdown  \
cost_scenario model_name                                      
0 bps         entry_state_calm_0                    -0.0471   
              entry_state_calm_0.5                  -0.0480   
              entry_state_calm_0.5_no_upgrade       -0.0474   
              entry_state_calm_0_no_upgrade         -0.0489   
              entry_state_calm_1                    -0.0497   
              ewma_z_band                           -0.0534   
              naive_zscore                          


Entry-state model, point-in-time pairs: mean and median yearly fold metrics


total_return          \
                                                      mean  median   
cost_scenario model_name                                             
0 bps         entry_state_calm_0                    0.0142  0.0119   
              entry_state_calm_0.5                  0.0138  0.0136   
              entry_state_calm_0.5_no_upgrade       0.0114  0.0122   
              entry_state_calm_0_no_upgrade         0.0118  0.0098   
              entry_state_calm_1                    0.0152  0.0163   
              ewma_z_band                           0.0169  0.0149   
              naive_zscore                          0.0162  0.0128   
5 bps         entry_state_calm_0                    0.0111  0.0098   
              entry_state_calm_0.5                  0.0106  0.0113   
              entry_state_calm_0.5_no_upgrade       0.0090  0.0098   
              entry_state_calm_0_no_upgrade         0.0100  0.0078   
              entry_state_calm_1                    0.0115  0.0131   
              ewma_z_band                           0.0123  0.0119   
              naive_zscore                          0.0085  0.0065   

                                              annualized_return          \
                                                           mean  median   
cost_scenario model_name                                                  
0 bps         entry_state_calm_0                         0.0142  0.0119   
              entry_state_calm_0.5                       0.0138  0.0137   
              entry_state_calm_0.5_no_upgrade            0.0114  0.0122   
              entry_state_calm_0_no_upgrade              0.0118  0.0098   
              entry_state_calm_1                         0.0153  0.0163   
              ewma_z_band                                0.0169  0.0149   
              naive_zscore                               0.0162  0.0128   
5 bps         entry_state_calm_0                         0.0112  0.0098   
              entry_state_calm_0.5                       0.0106  0.0113   
              entry_state_calm_0.5_no_upgrade            0.0090  0.0098   
              entry_state_calm_0_no_upgrade              0.0100  0.0078   
              entry_state_calm_1                         0.0115  0.0131   
              ewma_z_band                                0.0123  0.0119   
              naive_zscore                               0.0085  0.0065   

                                              sharpe_ratio          \
                                                      mean  median   
cost_scenario model_name                                             
0 bps         entry_state_calm_0                    0.4929  0.3588   
              entry_state_calm_0.5                  0.4538  0.3856   
              entry_state_calm_0.5_no_upgrade       0.3837  0.4000   
              entry_state_calm_0_no_upgrade         0.7753  0.7243   
              entry_state_calm_1                    0.4477  0.3562   
              ewma_z_band                           0.4382  0.4818   
              naive_zscore                          0.4551  0.3619   
5 bps         entry_state_calm_0                    0.4027  0.3103   
              entry_state_calm_0.5                  0.3606  0.3114   
              entry_state_calm_0.5_no_upgrade       0.3038  0.3075   
              entry_state_calm_0_no_upgrade         0.6988  0.6347   
              entry_state_calm_1                    0.3453  0.3072   
              ewma_z_band                           0.3130  0.4017   
              naive_zscore                          0.2650  0.2231   

                                              sortino_ratio          \
                                                       mean  median   
cost_scenario model_name                                              
0 bps         entry_state_calm_0                     1.1848  0.5231   
              entry_state_calm_0.5                   0.9512  0.6068   
              entry_state_calm_0.

... (4 outputs left out)


Per-trade gross P&L by state at entry (all years):


,trades,avg_days,avg_pnl_bps,median_pnl_bps,win_rate
state,,,,,
calm throughout,19,26.0,18.5,56.5,0.74
"calm, upgraded later",57,80.0,22.3,60.1,0.68
favourable at entry,58,63.0,105.4,116.9,0.86



Excluding 2020 and 2022:


,trades,avg_days,avg_pnl_bps,median_pnl_bps,win_rate
state,,,,,
calm throughout,16,28.0,39.6,57.8,0.81
"calm, upgraded later",47,89.0,-13.2,43.3,0.62
favourable at entry,37,61.0,89.6,106.9,0.86
